create widgets and variables for these two : landing_timestamp, table_name, add two more widgets and variables for merge_keys and load_type 

In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("STEP 1: Reading notebook parameters")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'")
print()

In [0]:
source_table = f"project1_gs_caresync.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading data from Bronze")
print("=" * 80)
print(f"  Source table     : {source_table}")
print(f"  Filter condition : {filter_condition}")
print("  Equivalent query :")
print(f"    SELECT * FROM {source_table}\n    WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"  Records read from Bronze : {bronze_count}")
print(f"  Columns ({len(bronze_df.columns)}) : {bronze_df.columns}")
if bronze_count == 0:
    print("  WARNING: No records found for this landing_timestamp.")
print()


In [0]:
%skip
bronze_df.display()

add two more columns to bronze_df. the columns are insert_timestamp as current_timestamp and update_timestamp as current timestamp name it as updated_df

In [0]:
from pyspark.sql.functions import current_timestamp

print("=" * 80)
print("STEP 3: Adding audit columns")
print("=" * 80)
print("  Adding column 'insert_timestamp' = current_timestamp()")
print("  Adding column 'update_timestamp' = current_timestamp()")

updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

print(f"  updated_df columns ({len(updated_df.columns)}) : {updated_df.columns}")
print()

In [0]:
%skip
display(updated_df)

In [0]:
from delta.tables import DeltaTable

target_table = f"project1_gs_caresync.silver.{table_name}"
load_type_norm = load_type.strip().upper()

print("=" * 80)
print("STEP 4: Writing data to Silver")
print("=" * 80)
print(f"  Target table        : {target_table}")
print(f"  Load type (input)   : '{load_type}'")
print(f"  Load type (normal.) : '{load_type_norm}'")
print(f"  Target table exists : {spark.catalog.tableExists(target_table)}")

if load_type_norm == "FULL":
    # Full load: overwrite target table
    print("  Mode: FULL -> overwriting the target table with incoming data")
    print("  Equivalent query :")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS\n    SELECT * FROM <updated_df>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("  FULL load completed successfully.")

elif load_type_norm == "APPEND":
    # Append: add incoming records to target table (creates it if missing)
    print("  Mode: APPEND -> appending incoming records to the target table (creates it if missing)")
    print("  Equivalent query :")
    print(f"    INSERT INTO {target_table}\n    SELECT * FROM <updated_df>")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("  APPEND load completed successfully.")

elif load_type_norm == "MERGE":
    # Merge: update existing records and insert new ones based on merge keys
    print("  Mode: MERGE -> update existing records and insert new ones based on merge keys")
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"  Parsed merge keys : {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type = 'MERGE'")

    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")
    print("  All merge keys are present in source data.")

    if not spark.catalog.tableExists(target_table):
        # First run: create the table
        print("  Target table does not exist -> first run, creating it from incoming data")
        print("  Equivalent query :")
        print(f"    CREATE TABLE {target_table} AS\n    SELECT * FROM <updated_df>")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("  Target table created successfully.")
    else:
        # One source row per key, otherwise MERGE fails on multiple matches
        print("  Target table exists -> performing MERGE")
        source_df = updated_df.dropDuplicates(keys)
        print(f"  Source rows deduplicated on keys {keys}")

        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])

        # On update keep the original insert_timestamp; refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        print(f"  Merge condition   : {merge_condition}")
        print(f"  Columns to UPDATE : {list(update_set.keys())} (insert_timestamp preserved)")
        print(f"  Columns to INSERT : {list(insert_set.keys())}")

        update_clause = ",\n        ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols = ", ".join(insert_set.keys())
        insert_vals = ", ".join(insert_set.values())
        merge_sql_preview = (
            f"MERGE INTO {target_table} AS t\n"
            f"USING <updated_df deduplicated on {keys}> AS s\n"
            f"ON {merge_condition}\n"
            f"WHEN MATCHED THEN UPDATE SET\n        {update_clause}\n"
            f"WHEN NOT MATCHED THEN INSERT ({insert_cols})\n"
            f"VALUES ({insert_vals})"
        )
        print("  Equivalent MERGE query being executed:")
        print("-" * 80)
        print(merge_sql_preview)
        print("-" * 80)

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute())

        # Print merge metrics from the latest Delta operation
        last_op = DeltaTable.forName(spark, target_table).history(1).select("operation", "operationMetrics").collect()[0]
        metrics = last_op["operationMetrics"] or {}
        print(f"  MERGE completed successfully (operation: {last_op['operation']}).")
        print(f"    Rows inserted : {metrics.get('numTargetRowsInserted', 'n/a')}")
        print(f"    Rows updated  : {metrics.get('numTargetRowsUpdated', 'n/a')}")
        print(f"    Source rows   : {metrics.get('numSourceRows', 'n/a')}")

else:
    print(f"  ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected FULL, APPEND or MERGE.")

print()


In [0]:
%skip
%sql
select * from project1_gs_caresync.silver.hospitals

provide record_count as output as notebook.exit, take count from updated_df 

In [0]:
print("=" * 80)
print("STEP 5: Returning record count")
print("=" * 80)
record_count = updated_df.count()
print(f"  record_count = {record_count}")
print("  Exiting notebook with record_count as output.")
dbutils.notebook.exit(str(record_count))

This prompt is need to run for enitre notebook. So ask Genie at right top corner.

this notebook is currently performs full load and overwrite of data. this is based on load_type ='FULL'. Now I want to implement 2 other load_types 1. 'APPEND', and 2. 'MERGE'. if it is append then simply append the data to target table. if it is merge then insert new records and update existing records based on merge keys.

This code is working fine. put some detailed print statement so that I can know what is happening at each step. print the queries as well that is being prepared 